# 01 · Meta descriptions with BERT extractive summarisation (the 2021 method, refreshed)

Companion notebook to the MLforSEO tutorial *Automatically Generate Meta Descriptions Using Python: from BERT to LLMs* by Lazarina Stoy.

This is the original 2021 approach: take the text of each page, run Google's BERT through the
[`bert-extractive-summarizer`](https://github.com/dmmiller612/bert-extractive-summarizer) library, and keep the sentence(s)
that best represent the page. **Extractive** means it never writes new words. It only picks existing sentences, so it can't invent facts.

**What changed in the refresh**
- `error_bad_lines=False` was removed in pandas 2.0, so we use `on_bad_lines="skip"`.
- The model is now loaded **once**, not once per URL (the 2021 loop reloaded BERT for every page).
- `max_length=150` never limited the *output* length. In this library, `min_length`/`max_length` filter *which sentences are allowed in*, by character count.
  The size of the summary comes from `ratio` (default 0.2 = 20% of sentences) or `num_sentences`. That's why the 2021 output was often several sentences long.
- We now ask for **one sentence**, set `use_first=False` (by default the library always keeps the page's first sentence), and flag anything too long or too short.
- Option B scrapes pages for you with `trafilatura`, if you don't have a crawler export.

**Runtime:** CPU is fine (around 2 to 15 seconds per page). Colab: Runtime → Change runtime type → T4 GPU makes it faster.

In [1]:
# Install anything that's missing (works in Google Colab and locally)
import importlib.util, subprocess, sys

PACKAGES = {'summarizer': 'bert-extractive-summarizer', 'trafilatura': 'trafilatura', 'pandas': 'pandas', 'torch': 'torch', 'transformers': 'transformers'}  # import name -> pip name
missing = [pip for mod, pip in PACKAGES.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Ready. Installed now:", missing or "nothing, all present")

Ready. Installed now: nothing, all present


## 1. Load your pages
The file format from the original tutorial still works: **CSV, column A = URL, column B = scraped content**. A sample `inputdata.csv` (five MLforSEO pages) ships with this notebook.

In [2]:
import pandas as pd

# OPTION A (the 2021 way): a CSV you exported from Screaming Frog, Sitebulb, etc.
#   Column A = URL, Column B = the page's main content. Title and Top query are optional extras.
INPUT_FILE = "inputdata.csv"

# OPTION B: no crawler export? Put one URL per line in urls.txt and set this to True.
#   trafilatura downloads each page and pulls out the main text and the title for you.
SCRAPE_FROM_URL_LIST = False
URL_LIST_FILE = "urls.txt"

# In Colab, upload your file first (folder icon on the left), or uncomment:
# from google.colab import files; files.upload()

if SCRAPE_FROM_URL_LIST:
    import trafilatura
    rows = []
    for url in [u.strip() for u in open(URL_LIST_FILE) if u.strip()]:
        html = trafilatura.fetch_url(url)
        text = trafilatura.extract(html) if html else None
        meta = trafilatura.extract_metadata(html) if html else None
        rows.append({"URL": url, "Content": text or "", "Title": getattr(meta, "title", "") or ""})
    df = pd.DataFrame(rows)
else:
    # error_bad_lines=False (2021) was removed in pandas 2.0; on_bad_lines="skip" replaces it
    df = pd.read_csv(INPUT_FILE, encoding="utf-8-sig", on_bad_lines="skip")
    df = df.rename(columns={df.columns[0]: "URL", df.columns[1]: "Content"})

for col in ["Title", "Top query"]:
    if col not in df.columns:
        df[col] = ""
df = df.fillna("")
df["Words"] = df["Content"].str.split().str.len()
df[["URL", "Title", "Top query", "Words"]]

,URL,Title,Top query,Words
0,https://www.mlforseo.com/machine-learning-tuto...,"What Fuzzy Matching Is, and Where It Fits in SEO",what is fuzzy matching,2349
1,https://www.mlforseo.com/machine-learning-tuto...,Topic Modelling for SEO with BERTopic,bertopic seo,1571
2,https://www.mlforseo.com/machine-learning-impl...,How to do keyword clustering with KeyBERT,keybert keyword clustering,2212
3,https://www.mlforseo.com/machine-learning-impl...,How to Automatically Optimize your SEO Metadat...,automate meta descriptions,1938
4,https://www.mlforseo.com/glossary-term/bert-bi...,BERT (Bidirectional Encoder Representations fr...,what is bert,233


## 2. Load BERT once

`Summarizer()` downloads `bert-large-uncased` the first time (about 1.3 GB), the same model the 2021 tutorial used.
If that's slow on your machine, `Summarizer(model="distilbert-base-uncased")` is a smaller, faster alternative.

In [3]:
from summarizer import Summarizer

model = Summarizer()   # loaded ONCE, outside the loop

/private/tmp/claude-501/-Users-lazarinastoy--claude-spares-spare-u7uWn6/ae32973c-ea32-48ff-9f52-1fc9b67f7ff3/scratchpad/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/private/tmp/claude-501/-Users-lazarinastoy--claude-spares-spare-u7uWn6/ae32973c-ea32-48ff-9f52-1fc9b67f7ff3/scratchpad/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. The 2021 call, as published

This is the exact call from the original post: `model(body, max_length=150)`. Look at the character counts. Most outputs are far longer
than a meta description can be, because `max_length` limits the sentences that are *considered*, not the length of the result.

In [4]:
df["BERT 2021 call"] = [model(str(body), max_length=150) for body in df["Content"]]
df["BERT 2021 chars"] = df["BERT 2021 call"].str.len()
df[["URL", "BERT 2021 chars", "BERT 2021 call"]]

,URL,BERT 2021 chars,BERT 2021 call
0,https://www.mlforseo.com/machine-learning-tuto...,663,"Redirect maps, 404 clean-ups, hreflang audits,..."
1,https://www.mlforseo.com/machine-learning-tuto...,573,For most of the text-heavy work SEOs actually ...
2,https://www.mlforseo.com/machine-learning-impl...,813,"In this guide, I’ll show you how to work with ..."
3,https://www.mlforseo.com/machine-learning-impl...,1419,If you’re looking to enhance your website’s SE...
4,https://www.mlforseo.com/glossary-term/bert-bi...,138,"BERT is a language model developed by Google, ..."


## 4. The refreshed call: one sentence, sensible length

- `num_sentences=1` returns a single sentence.
- `min_length=60, max_length=155` only lets sentences between 60 and 155 characters compete, so whatever wins already fits the SERP.
- `use_first=False` stops the library from always handing you the first sentence of the page. Try both settings, because on well-written
  pages the intro sentence is often the best summary anyway.

In [5]:
def bert_meta(body, use_first=False):
    body = str(body)
    if len(body.split()) < 40:
        return ""   # too little text to summarise; write these by hand (and consider whether the page is thin)
    return model(body, num_sentences=1, min_length=60, max_length=155, use_first=use_first).strip()

df["BERT meta (use_first=False)"] = [bert_meta(b, use_first=False) for b in df["Content"]]
df["BERT meta (use_first=True)"]  = [bert_meta(b, use_first=True)  for b in df["Content"]]
df[["URL", "BERT meta (use_first=False)", "BERT meta (use_first=True)"]]

,URL,BERT meta (use_first=False),BERT meta (use_first=True)
0,https://www.mlforseo.com/machine-learning-tuto...,"Redirect maps, 404 clean-ups, hreflang audits,...","Redirect maps, 404 clean-ups, hreflang audits,..."
1,https://www.mlforseo.com/machine-learning-tuto...,"A quick, important caveat: this is a very limi...",For most of the text-heavy work SEOs actually ...
2,https://www.mlforseo.com/machine-learning-impl...,This feature allows you to visually emphasize ...,"In this guide, I’ll show you how to work with ..."
3,https://www.mlforseo.com/machine-learning-impl...,The main idea is to ensure that your page titl...,If you’re looking to enhance your website’s SE...
4,https://www.mlforseo.com/glossary-term/bert-bi...,BERT serves as a drop-in replacement for earli...,"BERT is a language model developed by Google, ..."


## 5. Quality checks
Automated text still needs a human pass. These checks flag the obvious problems: length, missing top query, duplicates.

In [6]:
import re

MIN_CHARS, MAX_CHARS = 70, 155   # Google truncates by pixel width (~920px desktop); ~155 characters is a safe proxy

def check_description(desc, source_text, top_query=""):
    """Flags problems a human should look at. It does not prove a description is good."""
    flags = []
    n = len(desc)
    if n == 0:
        flags.append("EMPTY")
    elif n < MIN_CHARS:
        flags.append(f"SHORT ({n})")
    elif n > MAX_CHARS:
        flags.append(f"LONG ({n})")
    # numbers that appear in the description but nowhere in the page = classic hallucination
    src_numbers = set(re.findall(r"\d+(?:[.,]\d+)?", source_text))
    new_numbers = [x for x in re.findall(r"\d+(?:[.,]\d+)?", desc) if x not in src_numbers]
    if new_numbers:
        flags.append("NUMBER NOT IN PAGE: " + ", ".join(new_numbers))
    # capitalised words (names, brands, tools) that never appear in the page
    src_lower = source_text.lower()
    new_names = [w for w in re.findall(r"\b[A-Z][A-Za-z0-9]{2,}\b", desc)[1:] if w.lower() not in src_lower]
    if new_names:
        flags.append("NAME NOT IN PAGE: " + ", ".join(sorted(set(new_names))))
    if top_query:
        q_words = [w for w in re.findall(r"\w+", top_query.lower()) if len(w) > 3]
        if q_words and not any(w in desc.lower() for w in q_words):
            flags.append("TOP QUERY MISSING")
    return "; ".join(flags) or "OK"

In [7]:
# Duplicate descriptions are what we're trying to get rid of in the first place, so check for them
def flag_duplicates(frame, col):
    dupes = frame[col].str.strip().str.lower().duplicated(keep=False) & frame[col].str.strip().ne("")
    frame.loc[dupes, col + " check"] = frame.loc[dupes, col + " check"].replace("OK", "") + "; DUPLICATE"
    frame[col + " check"] = frame[col + " check"].str.strip("; ")
    return frame

In [8]:
for col in ["BERT meta (use_first=False)", "BERT meta (use_first=True)"]:
    df[col + " check"] = [check_description(d, s, q) for d, s, q in zip(df[col], df["Content"], df["Top query"])]
    df = flag_duplicates(df, col)

df[["URL", "BERT meta (use_first=False)", "BERT meta (use_first=False) check",
    "BERT meta (use_first=True)", "BERT meta (use_first=True) check"]]

,URL,BERT meta (use_first=False),BERT meta (use_first=False) check,BERT meta (use_first=True),BERT meta (use_first=True) check
0,https://www.mlforseo.com/machine-learning-tuto...,"Redirect maps, 404 clean-ups, hreflang audits,...",TOP QUERY MISSING,"Redirect maps, 404 clean-ups, hreflang audits,...",TOP QUERY MISSING
1,https://www.mlforseo.com/machine-learning-tuto...,"A quick, important caveat: this is a very limi...",OK,For most of the text-heavy work SEOs actually ...,TOP QUERY MISSING
2,https://www.mlforseo.com/machine-learning-impl...,This feature allows you to visually emphasize ...,OK,"In this guide, I’ll show you how to work with ...",OK
3,https://www.mlforseo.com/machine-learning-impl...,The main idea is to ensure that your page titl...,OK,If you’re looking to enhance your website’s SE...,OK
4,https://www.mlforseo.com/glossary-term/bert-bi...,BERT serves as a drop-in replacement for earli...,OK,"BERT is a language model developed by Google, ...",OK


## 6. Save
`output.csv` lands next to this notebook (in Colab: the folder icon on the left).

In [9]:
df.drop(columns=["Content"]).to_csv("output_bert.csv", index=False)
print("Saved output_bert.csv with", len(df), "rows")

Saved output_bert.csv with 5 rows


## What to expect

Extractive summaries are safe (every word comes from your page) but they are **not written for clicks**. You'll often get a sentence that is accurate
but a bit flat, or one that leans on context ("This is why…"). Treat the output as a first draft: add the top query where it fits, tighten the wording,
and check the flags. Notebook 02 shows the modern options.